<a href="https://colab.research.google.com/github/2403a52276-ctrl/NLP/blob/main/Lab12_2_TextCNN_MultiFilter_N_Sreeshanth_2403A52276_B10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**STEP 1 — Create Notebook / Script**

**File Name :**

Lab12.2_TextCNN_MultiFilter_N Sreeshanth_2403A52276

In [ ]:
# Install Gensim for Word2Vec
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 41.2 MB/s eta 0:00:00


**STEP 2 — Import Libraries**

In [ ]:
# Core numerical library
import numpy as np

# Deep learning (Keras - TensorFlow backend)
from tensorflow.keras import Model
from tensorflow.keras.layers import (
    Input,
    Embedding,
    Conv1D,
    GlobalMaxPooling1D,
    Dense,
    Concatenate
)

# Text preprocessing utilities
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Word embedding model
from gensim.models import Word2Vec as W2V

**STEP 3 — Load and Explore Dataset**

In [ ]:
# Updated sample dataset
texts = [
    "this product is amazing",
    "i really like this item",
    "this product is awful",
    "i dislike this item",
    "excellent quality",
    "poor performance"
]

# Labels: 1 = positive, 0 = negative
labels = [1, 1, 0, 0, 1, 0]


**STEP 5 — Vocabulary and Encoding**


In [ ]:
# Initialize tokenizer object
tok = Tokenizer()

# Fit tokenizer on input text data
tok.fit_on_texts(texts)

# Transform text into integer sequences
seq_data = tok.texts_to_sequences(texts)

# Extract word-to-index mapping
vocab_dict = tok.word_index

# Display vocabulary
print(vocab_dict)

{'this': 1, 'product': 2, 'is': 3, 'i': 4, 'item': 5, 'amazing': 6, 'really': 7, 'like': 8, 'awful': 9, 'dislike': 10, 'excellent': 11, 'quality': 12, 'poor': 13, 'performance': 14}



**STEP 4 — Text Preprocessing**


In [ ]:
print(seq_data)

[[1, 2, 3, 6], [4, 7, 8, 1, 5], [1, 2, 3, 9], [4, 10, 1, 5], [11, 12], [13, 14]]


In [ ]:
# Define maximum sequence length
max_len = 6

# Apply padding to make all sequences equal length
X_data = pad_sequences(seq_data, maxlen=max_len)

# Convert labels into numpy array
y_data = np.asarray(labels)

________________________________________
**STEP 6 — Train–Test Split**



In [ ]:
X

array([[ 0,  0,  1,  2,  3,  6],
       [ 0,  4,  7,  8,  1,  5],
       [ 0,  0,  1,  2,  3,  9],
       [ 0,  0,  4, 10,  1,  5],
       [ 0,  0,  0,  0, 11, 12],
       [ 0,  0,  0,  0, 13, 14]], dtype=int32)

In [ ]:
# Split each sentence into list of words
token_lists = [line.split() for line in texts]

# Train Word2Vec model
w2v = W2V(
    sentences=token_lists,
    vector_size=50,
    window=3,
    min_count=1
)

In [ ]:
# Calculate vocabulary length and embedding size
vocab_length = len(vocab_dict) + 1
vec_size = 50

# Create empty embedding matrix
embed_matrix = np.zeros((vocab_length, vec_size))

# Assign Word2Vec vectors to corresponding indices
for term, i in vocab_dict.items():
    if term in w2v.wv:
        embed_matrix[i] = w2v.wv[term]

In [ ]:
embed_matrix[1]

array([-1.07245450e-03,  4.72862710e-04,  1.02066994e-02,  1.80185456e-02,
       -1.86058991e-02, -1.42336180e-02,  1.29177449e-02,  1.79459769e-02,
       -1.00308564e-02, -7.52674323e-03,  1.47610093e-02, -3.06694279e-03,
       -9.07322671e-03,  1.31081035e-02, -9.72032081e-03, -3.63203534e-03,
        5.75315952e-03,  1.98374758e-03, -1.65704302e-02, -1.88976359e-02,
        1.46235321e-02,  1.01405242e-02,  1.35153867e-02,  1.52573106e-03,
        1.27017805e-02, -6.81073172e-03, -1.89280277e-03,  1.15371468e-02,
       -1.50432754e-02, -7.87220709e-03, -1.50231645e-02, -1.86008448e-03,
        1.90762375e-02, -1.46383336e-02, -4.66753729e-03, -3.87548213e-03,
        1.61548741e-02, -1.18617918e-02,  9.03248801e-05, -9.50746797e-03,
       -1.92071013e-02,  1.00145862e-02, -1.75191704e-02, -8.78365058e-03,
       -7.01999670e-05, -5.92362892e-04, -1.53224804e-02,  1.92294866e-02,
        9.96411592e-03,  1.84662864e-02])

**STEP 7 — Build Multi-Filter 1D CNN Model**

In [ ]:
# Define input layer with sequence length
inp_layer = Input(shape=(max_len,))

In [ ]:
# Create embedding layer using pre-trained Word2Vec matrix
embed_layer = Embedding(
    input_dim=vocab_length,
    output_dim=vec_size,
    weights=[embed_matrix],   # pre-trained embeddings
    input_length=max_len,
    trainable=False           # freeze embeddings
)(inp_layer)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
# Apply multiple convolution filters with different kernel sizes
conv_layer1 = Conv1D(100, 3, activation='relu')(embed_layer)
conv_layer2 = Conv1D(100, 4, activation='relu')(embed_layer)
conv_layer3 = Conv1D(100, 5, activation='relu')(embed_layer)

In [ ]:
# Apply global max pooling to each convolution output
p_layer1 = GlobalMaxPooling1D()(conv_layer1)
p_layer2 = GlobalMaxPooling1D()(conv_layer2)
p_layer3 = GlobalMaxPooling1D()(conv_layer3)

In [ ]:
# Combine pooled features from all convolution layers
merged_layer = Concatenate()([p_layer1, p_layer2, p_layer3])

In [ ]:
# Fully connected layer for feature learning
fc_layer = Dense(10, activation='relu')(merged_layer)


In [ ]:
# Final output layer for binary classification
out_layer = Dense(1, activation='sigmoid')(fc_layer)

________________________________________
**STEP 8 — Model Training**



In [ ]:
# Build the CNN model by connecting input and output layers
cnn_model = Model(inputs=inp_layer, outputs=out_layer)

In [ ]:
# Compile the model with optimizer, loss function, and evaluation metric
cnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
# Display the architecture of the CNN model
cnn_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 6)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 6, 50)     │        750 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 4, 100)    │     15,100 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 3, 100)    │     20,100 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 2, 100)    │     25,100 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 100)       │          0 │ conv1d[0][0]      │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 100)       │          0 │ conv1d_1[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 100)       │          0 │ conv1d_2[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 300)       │          0 │ global_max_pooli… │
│ (Concatenate)       │                   │            │ global_max_pooli… │
│                     │                   │            │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 10)        │      3,010 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │         11 │ dense[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 64,071 (250.28 KB)

 Trainable params: 63,321 (247.35 KB)

 Non-trainable params: 750 (2.93 KB)

In [ ]:
# Train the CNN model on the dataset
cnn_model.fit(
    X_data,
    y_data,
    epochs=10,
    batch_size=2
)

Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.1667 - loss: 0.6969  
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.5000 - loss: 0.6912
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6667 - loss: 0.6890
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6667 - loss: 0.6872
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.6667 - loss: 0.6855 
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6667 - loss: 0.6833
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6667 - loss: 0.6814
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6667 - loss: 0.6788
Epoch 9/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6667 - loss: 0.6767
Epoch 10/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8333 - loss: 0.6736


**STEP 9 — Model Evaluation**

In [ ]:
# New sample text for testing
sample_text = ["this product is amazing"]

# Convert text to sequence
test_seq = tok.texts_to_sequences(sample_text)

# Apply padding
test_pad = pad_sequences(test_seq, maxlen=max_len)

# Predict using trained model
pred_result = cnn_model.predict(test_pad)

# Display prediction
print(pred_result)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 148ms/step
[[0.5134985]]


**STEP 10 — Result Analysis**

Multi-filter CNN models use different kernel sizes to capture various patterns in text, such as short phrases and longer contextual information. Smaller filters (like size 3) focus on local features, while larger filters (like 4 and 5) capture broader relationships between words. This helps the model learn richer and more meaningful representations of the input text. As a result, the classification performance improves compared to using a single filter size. Another strength of multi-filter CNNs is their ability to detect multiple types of features simultaneously,

**STEP 11 — Lab Report**

**Text Classification using Multi-Filter CNN**

This experiment implements text classification using a Multi-Filter 1D CNN model. The text data is preprocessed using tokenization, encoding, and padding to convert it into numerical form. Word2Vec is used to generate word embeddings, which capture semantic meaning of words. The CNN model uses multiple convolution filters of different sizes to extract various features from the text. These features are combined and passed through dense layers for classification. The model is trained using binary crossentropy loss and Adam optimizer. Finally, the trained model predicts the class of new input text. The use of multiple filters improves feature extraction and overall model performance.
